# 도메인 평가 에이전트 — 데이터센터 관점

KV cache 최적화 기술 두 건이 **데이터센터** 환경에서 각 요구사항 축마다 어떤 조건에서
적합하다고 평가받는지 웹 검색 기반 RAG로 근거를 모아 판단합니다.

- SW: DeepSeek-V2 MLA — 저차원 잠재 압축으로 KV cache 감소
- HW: ITME — CXL-Hybrid 계층 메모리로 추론 처리량 향상

우열을 판정하지 않고, 관점에 따라 평가가 어떻게 갈리는지를 근거와 함께 기록합니다.

```
질문 생성(한/영) → 웹 검색(출처 등급 필터) → 본문 수집
                      ├ 짧은 문서: 그대로 근거
                      └ 긴 문서: 청킹 → BM25+dense 색인 → 해당 부분만 근거
  → 적용 조건 검토 → 근거 충분성 ─(부족)→ 질의 보완 → 재검색
  → 적합성 분석 + 자체 품질 점검 (판정 12건 = 6축 × 2기술, self_check 하나로 통합)
```

**v3 변경점**: 결정적 guard/linter와 별도 LLM judge(코드 3개 파일)를 없애고, 분석
프롬프트 하나에 자체 검증(self_check)을 흡수했습니다. 코드에 남은 검증은 참조
무결성(존재하지 않는 근거 라벨 제거) 하나뿐입니다 — 이유와 트레이드오프는
`docs/DOMAIN_AGENT.md` 참조.

## 왜 데이터센터이고, 왜 임베딩인가

**데이터센터 고정.** KV cache는 연산 병목을 메모리 병목으로 바꾼 문제이고, 그 압박이
실제 비용이 되는 곳이 데이터센터입니다. 온디바이스에서는 한 사용자가 자기 메모리를 쓰고
끝나지만, 데이터센터에서는 한 요청의 KV cache가 다른 사용자 몫 HBM을 잠식해 동시 처리
요청 수를 떨어뜨립니다. 또 ITME가 전제하는 CXL은 랙 스케일 규격이라 온디바이스에 없어서,
SW 압축과 HW 확장이 같은 무대에서 비교되는 환경은 데이터센터뿐입니다.

**임베딩이 필요한 이유는 측정으로 확인했습니다.** 이 에이전트는 평가 질문을 한국어로
만드는데 근거 문서는 영어입니다. 키워드 검색(BM25)만으로 되는지 재봤습니다.

| 방식 | 한국어 질의 Hit@5 | 영어 질의 Hit@5 |
|---|---|---|
| BM25 | 0.29 (2/7) | 1.00 (9/9) |
| dense (bge-m3) | 0.86 (6/7) | 0.89 (8/9) |

BM25는 한국어 질의에서 무너집니다. 겹치는 어휘가 없으니 당연합니다. 교차언어 검색을
성립시키는 것이 임베딩이고, 동시에 영어 질의에서는 BM25가 더 나으므로 둘 다 씁니다.
자세한 근거는 `docs/DOMAIN_AGENT.md`, 재현은 `python -m agents.domain.tools.ablation` 참조.

In [1]:
# 노트북이 notebooks/ 안에 있으므로 레포 루트를 import 경로에 넣는다.
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env", override=True)

# TAVILY_API_KEY 가 없어도 동작한다. 그 경우 캐시를 재생하는 오프라인 목 모드로 떨어진다.
import os

assert os.environ.get("OPENAI_API_KEY"), ".env 에 OPENAI_API_KEY 가 필요합니다"
print("루트:", ROOT)
print("웹 검색 모드:", "Tavily" if os.environ.get("TAVILY_API_KEY") else "오프라인 목(캐시 재생)")

루트: /Users/sanlee/Desktop/SKALA/코딩파일/Ai-service/Capstone_Ai_RAG
웹 검색 모드: Tavily


## 1. 출처 정책

검색 제공자의 도메인 필터는 신뢰하지 않습니다. Tavily `include_domains` 에 13개를
지정했더니 medium·substack·youtube 가 그대로 반환되는 것을 확인했습니다(2개일 때는 정상).
그래서 결과를 받은 뒤 등급표로 다시 거르고, 거른 이유를 검색 로그에 남깁니다.
포럼·커뮤니티 서브도메인(`forums.developer.nvidia.com` 등)도 벤더 공식 문서와 구분해
따로 걸러냅니다 — 실측에서 이게 안 걸러져 근거 35건 중 19건이 포럼 글이었던 적이 있습니다.

In [2]:
from agents.domain.tools.websearch import SOURCE_TIERS, classify_source

# 등급이 곧 근거 강도다. 매체 보도만 근거인 주장은 basis=direct 로 올리지 않는다.
for tier, domains in SOURCE_TIERS.items():
    print(f"{tier:<10} {len(domains):>2}개  예: {', '.join(domains[:3])}")

print("\n등급 판정 예시")
for url in [
    "https://arxiv.org/abs/2405.04434",
    "https://www.nvidia.com/en-us/data-center/",
    "https://computeexpresslink.org/spec",
    "https://forums.developer.nvidia.com/t/x",
    "https://medium.com/@someone/kv-cache",
]:
    verdict = classify_source(url) or "거부"
    print(f"  {verdict:<14} {url}")

paper       9개  예: arxiv.org, ieee.org, acm.org
patent      3개  예: patents.google.com, patentscope.wipo.int, uspto.gov
standard    4개  예: computeexpresslink.org, jedec.org, opencompute.org
vendor     13개  예: nvidia.com, intel.com, amd.com
news       10개  예: reuters.com, bloomberg.com, theregister.com

등급 판정 예시
  paper          https://arxiv.org/abs/2405.04434
  vendor         https://www.nvidia.com/en-us/data-center/
  standard       https://computeexpresslink.org/spec
  거부             https://forums.developer.nvidia.com/t/x
  거부             https://medium.com/@someone/kv-cache


In [3]:
# 검색 제공자를 만든다. 질의 단위 파일 캐시를 쓰는 이유는 재현성이다.
# 같은 질의가 실행마다 다른 결과를 주면 ablation 비교도 보고서 재생성도 성립하지 않는다.
from agents.domain.tools.websearch import build_search_provider

search_provider = build_search_provider(ROOT / "data/search_cache")
print("제공자:", type(search_provider).__name__)

제공자: TavilyWebSearch


## 2. 입력 State (팀 공통 AppState)

부모 State는 `graph/state.py`의 `AppState` 하나입니다. 도메인 노드는 `selected_tech`,
`request.as_of`, 선행 단계인 `technical_findings`만 봅니다. 시장·이해관계자 관점의
중간 결론을 같이 넘기면 도메인 판단이 그쪽 결론에 물들기 때문에, `project_input()`이
입력을 추려서 서브그래프에 넘깁니다. 아래에 시장·이해관계자 결과를 일부러 넣어 두고,
마지막에 누수 여부를 확인합니다.

In [4]:
from graph.state import create_initial_state

state = create_initial_state(
    request={
        "as_of": "2026-09-22",
        "language": "ko",
        "scope": "kv_cache_datacenter",
        "max_search_rounds": 2,  # 최초 검색 포함. 근거가 부족하면 질의를 바꿔 1회 더
    },
    selected_tech={
        "sw": {
            "name": "DeepSeek-V2 MLA (Multi-head Latent Attention)",
            "short_name": "MLA", "technology": "sw",
            "approach": "저차원 잠재 압축으로 KV cache 크기 자체를 줄임",
            "source_ids": ["arxiv:2405.04434"],
            "selection_reason": "저차원 잠재 압축으로 KV cache 93.3% 감소",
        },
        "hw": {
            "name": "ITME (Inference Tiered Memory Expansion, CXL-Hybrid)",
            "short_name": "ITME", "technology": "hw",
            "approach": "CXL 기반 계층 메모리로 KV cache를 HBM 밖까지 확장",
            "source_ids": ["arxiv:2606.12556"],
            "selection_reason": "CXL-Hybrid 계층 메모리로 추론 처리량 1.80배 향상",
        },
    },
    corpus_manifest=[],
)

# 선행 기술 조사 에이전트 산출물 모사(팀원 담당, 아직 미구현이라 최소 형태로 대신 채움)
state["technical_findings"] = {
    "claims": [
        {"technology": "sw", "text": "MLA는 KV를 저차원 잠재 벡터로 압축해 캐시를 93.3% 줄인다고 보고됨"},
        {"technology": "hw", "text": "ITME는 CXL-Hybrid 계층 메모리로 처리량 1.80배를 보고"},
    ],
    "records": [
        {"technology": "sw", "criterion": "TRL", "value": "7", "assessment": "상용 적용",
         "findings": "공개 정보 기반 추정"},
        {"technology": "hw", "criterion": "TRL", "value": "4", "assessment": "연구 시연",
         "findings": "공개 정보 기반 추정"},
    ],
}
# 관점 격리 확인용 미끼. 결과에 이 문장이 나타나면 격리가 깨진 것이다.
state["market_findings"] = {"claims": [{"text": "시장 규모가 급성장 중이다"}]}
state["stakeholder_findings"] = {"claims": [{"text": "업계가 CXL을 반긴다"}]}

from agents.domain.node import project_input

print("서브그래프에 실제로 넘어가는 입력:")
for key, value in project_input(state).items():
    print(f"  {key}: {str(value)[:80]}")

서브그래프에 실제로 넘어가는 입력:
  sw_name: DeepSeek-V2 MLA (Multi-head Latent Attention)
  hw_name: ITME (Inference Tiered Memory Expansion, CXL-Hybrid)
  as_of_date: 2026-09-22
  max_search_rounds: 2
  technical_summary: - (sw) MLA는 KV를 저차원 잠재 벡터로 압축해 캐시를 93.3% 줄인다고 보고됨
- (hw) ITME는 CXL-Hybrid 계층 메모리


## 3. 실행

LLM(GPT)과 검색 제공자는 State 밖의 런타임 의존성으로 주입합니다.
State 에는 JSON 직렬화 가능한 값만 둡니다(체크포인터 저장 경계).

웹 검색 → 본문 수집 → 긴 문서 색인이 포함되어 몇 분 걸립니다.

In [5]:
from langchain.chat_models import init_chat_model

from agents.domain import DomainAgentDeps, make_node
from agents.domain.prompts import PROMPT_VERSION
from agents.domain.tools.manifest import LLMRunInfo, RunManifest, current_git_commit, file_checksum

MODEL = "gpt-4o"  # 팀 통일 모델(ISSUE.md 2-3). 이전 gpt-4o-mini 에서 변경했다.

# 실행 매니페스트: 이 결과가 어떤 조건에서 나왔는지 남겨야 보고서 수치를 재현할 수 있다.
manifest = RunManifest(run_id="domain-nb-001", git_commit=current_git_commit(ROOT))
manifest.record_llm(LLMRunInfo(
    provider="openai", model=MODEL, temperature=0.0, max_tokens=None, seed=None,
    prompt_version=PROMPT_VERSION,
))

deps = DomainAgentDeps(
    llm=init_chat_model(MODEL, model_provider="openai", temperature=0),
    search_provider=search_provider,
    embedding_model="BAAI/bge-m3",  # 긴 문서 색인에만 쓰인다
    fetch_cache_dir=ROOT / "data/fetch_cache",
)

out = make_node(deps)(state)
manifest.record_embedding(deps.embedding_run_info)

# AppState 에 돌려주는 키. 관점별로 분리돼 병렬 분기에서 충돌하지 않는다.
print("반환 키:", sorted(out.keys()))
findings = out["domain_findings"]

/Users/sanlee/Desktop/SKALA/코딩파일/Ai-service/Capstone_Ai_RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 45019.57it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 69311.22it/s]

반환 키: ['domain_findings', 'evidence_store', 'quality_by_perspective', 'run_meta', 'search_log_by_perspective']


In [6]:
# AppState 에는 오류·수집량을 담을 전용 최상위 필드가 없어 run_meta 에 관점별로 남긴다.
run_meta = out["run_meta"]["domain"]
print(f"status={findings['status']}  검색 라운드={run_meta['search_rounds_used']}  "
      f"수집={run_meta['pages_used']}/200p")
print(f"근거 {len(out['evidence_store'])}건 / 주장 {len(findings['claims'])}건 / "
      f"판정 {len(findings['records'])}건 (6축 x 2기술 = 최대 12건)")

if findings["gaps"]:
    print("\n공백(Gap) — 보고서 한계점 장으로 이어진다")
    for g in findings["gaps"][:8]:
        print(f"  [{g['technology']}] {g['criterion'][:30]}: {g['reason'][:60]}")
if run_meta["errors"]:
    print("\n오류(숨기지 않고 남긴다)")
    for e in run_meta["errors"][:5]:
        print(" -", e[:120])

status=partial  검색 라운드=2  수집=195/200p
근거 31건 / 주장 12건 / 판정 12건 (6축 x 2기술 = 최대 12건)

공백(Gap) — 보고서 한계점 장으로 이어진다
  [both] 지연시간(TTFT·TPOT): 근거 부족으로 판단하지 못함
  [both] 정확도 유지: 근거 부족으로 판단하지 못함
  [both] 인프라 도입 비용과 운영 부담: 근거 부족으로 판단하지 못함

오류(숨기지 않고 남긴다)
 - 본문 수집 실패(https://openreview.net/pdf?id=JHvS2Q9RtW): HTTPStatusError: Client error '403 Forbidden' for url 'https://openr
 - 본문 수집 실패(https://openreview.net/pdf/72880f88aa934ab29407620344b11d7e3): HTTPStatusError: Client error '403 Forbidden' fo
 - 본문 수집 실패(https://dl.acm.org/doi/full/10.1145/3728636): HTTPStatusError: Client error '403 Forbidden' for url 'https://dl
 - 본문 수집 실패(https://dl.acm.org/doi/10.1145/3639564): HTTPStatusError: Client error '403 Forbidden' for url 'https://dl.acm.
 - 본문 수집 실패(https://dl.acm.org/doi/pdf/10.1145/3757347.3759131): HTTPStatusError: Client error '403 Forbidden' for url 'htt


## 4. 자체 품질 점검 (v3: 단일 프롬프트)

결정적 guard·표현 linter·별도 LLM judge(코드 3개 파일)를 없애고, 분석 프롬프트 하나가
스스로 인용 검증·수치 검증·표현 검증·균형 검증·커버리지를 점검해 `quality_by_perspective`
로 보고합니다. 코드에 남은 것은 참조 무결성(존재하지 않는 근거 라벨 제거)뿐입니다.

**트레이드오프**: 실행해보니 이 방식에서도 실제 문제가 나왔습니다 — 근거 하나가 경쟁
제품("Kimi K3")에 대한 뉴스였는데 이걸 HW(ITME) 판정에 유추 근거로 썼습니다. basis를
`inferred`로 낮추고 `limitations`에 실제 출처를 적어 정직하게 표시하긴 했지만, `findings`
문장만 보면 그 사실이 안 드러났습니다. 프롬프트에 "다른 제품 근거를 쓸 때는 findings에도
실제 대상을 명시하라"는 규칙을 추가해 개선했습니다(v3.1, `docs/DOMAIN_AGENT.md` 참조).

In [7]:
q = out["quality_by_perspective"]["domain"]
print(f"status={q['status']}")
print(f"violations({len(q['violations'])}):")
for v in q["violations"]:
    print("  -", v)
print(f"warnings({len(q['warnings'])}):")
for w in q["warnings"]:
    print("  -", w)
print(f"검사된 주장: {len(q['checked_claim_ids'])}건")

status=needs_review
violations(2):
  - 근거 없는 주장: hw-accuracy-1
  - 근거 없는 주장: sw-power-1
warnings(1):
  - sw와 hw의 서술 분량이 균형을 이루지 않음
검사된 주장: 12건


In [8]:
# records: 요구사항 축 하나 x 기술 하나마다 하나씩. 종합 에이전트가 상충을 기계적으로
# 찾도록 assessment/basis 를 열거형으로 고정한다.
for r in findings["records"]:
    value_part = f" ({r['value']})" if r["value"] else ""
    print(f"[{r['technology']}] {r['criterion'][:40]:<42} "
          f"basis={r['basis']:<10} scope={r['scope']:<7} {r['assessment']}{value_part}")
    print(f"    {r['findings'][:110]}")
    for lim in r["limitations"][:2]:
        print(f"    제약: {lim}")

[sw] HBM 용량 압박 완화: 요청당 KV cache 점유가 동시 처리 요청    basis=direct     scope=direct  suitable (93.3%)
    DeepSeek-V2 MLA는 KV를 저차원 잠재 벡터로 압축해 캐시를 93.3% 줄인다.
    제약: Non-standard implementations may limit adoption
[hw] HBM 용량 압박 완화: 요청당 KV cache 점유가 동시 처리 요청    basis=inferred   scope=class   conditional
    ITME는 CXL-Hybrid 계층 메모리로 HBM 용량 압박을 완화한다.
    제약: PCIe Gen4-based architecture limits bandwidth
[sw] 처리량(tokens/s)과 동시 요청 수: 멀티테넌시 환경에서 GPU 1   basis=direct     scope=direct  suitable
    DeepSeek-V2 MLA는 vLLM 프레임워크를 사용해 실질적인 추론 속도 향상을 달성한다.
    제약: Specific hardware setups may not be standard
[hw] 처리량(tokens/s)과 동시 요청 수: 멀티테넌시 환경에서 GPU 1   basis=direct     scope=direct  suitable (1.80x)
    ITME는 CXL-Hybrid 계층 메모리로 처리량을 1.80배 증가시킨다.
    제약: Specific hardware configurations
[sw] 지연시간(TTFT·TPOT): 서비스 SLA를 만족해야 하며 메모리 계층   basis=inferred   scope=class   conditional
    DeepSeek-V2 MLA는 KV 캐시 압축으로 지연시간을 줄일 수 있다.
    제약: Experiments use specific context lengths
[hw] 지연시간(TTFT·TPOT): 서비스 SLA를 만

In [9]:
# claims: 팀 스키마는 신뢰도(basis)를 Claim 이 아니라 VerdictRecord 에 둔다.
# 여기서는 각 주장이 어느 근거를 인용했는지만 확인한다.
for cl in findings["claims"]:
    print(f"[{cl['claim_id']}] ({cl['technology']}) {cl['text'][:90]}")
    print(f"  근거: {cl['evidence_ids']}")
    if cl["conditions"]:
        print(f"  전제: {cl['conditions'][:2]}")
    if cl["limitations"]:
        print(f"  한계: {cl['limitations'][:2]}")
    print()

[domain:claim:001] (sw) DeepSeek-V2 MLA는 KV를 저차원 잠재 벡터로 압축해 캐시를 93.3% 줄인다.
  근거: ['domain:ev:49f8b97e74a7']
  전제: ['KV compression dimension set to 512']
  한계: ['Non-standard implementations may limit adoption']

[domain:claim:002] (hw) ITME는 CXL-Hybrid 계층 메모리로 HBM 용량 압박을 완화한다.
  근거: ['domain:ev:c42789b8ef9b', 'domain:ev:03c2b25d5728']
  전제: ['CXL-hybrid memory architecture']
  한계: ['PCIe Gen4-based architecture limits bandwidth']

[domain:claim:003] (sw) DeepSeek-V2 MLA는 vLLM 프레임워크를 사용해 실질적인 추론 속도 향상을 달성한다.
  근거: ['domain:ev:3c08d7e8eb4b']
  전제: ['vLLM framework used']
  한계: ['Specific hardware setups may not be standard']

[domain:claim:004] (hw) ITME는 CXL-Hybrid 계층 메모리로 처리량을 1.80배 증가시킨다.
  근거: ['domain:ev:c42789b8ef9b']
  전제: ['CXL-hybrid memory architecture']
  한계: ['Specific hardware configurations']

[domain:claim:005] (sw) DeepSeek-V2 MLA는 KV 캐시 압축으로 지연시간을 줄일 수 있다.
  근거: ['domain:ev:bb347597fa13']
  전제: ['Longer contexts lead to higher speedups']
  한계: ['Experiments use specific

In [10]:
# evidence_store: 팀 graph.state.Evidence 형식. quote/page_or_locator 가 있어야
# 보고서 REFERENCE 장과 인용 추적이 가능하다.
from collections import Counter

print("출처 등급 분포:", Counter(e["source_type"] for e in out["evidence_store"].values()))
print("primary/secondary:", Counter(e["primary_or_secondary"] for e in out["evidence_store"].values()))
for ev in list(out["evidence_store"].values())[:3]:
    print(f"\n[{ev['id']}] ({ev['source_type']}, {ev['direct_or_proxy']}) {ev['page_or_locator']}")
    print(f"  {ev['title'][:70]}")
    print(f"  {ev['url'][:80]}")
    print(f"  인용: {ev['quote'][:110]}...")

출처 등급 분포: Counter({'paper': 27, 'vendor': 4})
primary/secondary: Counter({'primary': 27, 'secondary': 4})

[domain:ev:1b3da3e5f0f7] (paper, direct) chars:0-146
  Verifying your browser | OpenReview
  https://openreview.net/forum?id=TcVCu2PKb9
  인용: Complete the check below to continue to OpenReview Please complete the verification above. Have an OpenReview ...

[domain:ev:49f8b97e74a7] (paper, direct) chars:21000-24000+0
  DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-Experts La
  https://arxiv.org/html/2405.04434v2
  인용: set the number of attention heads n h n_{h} to 128 and the per-head dimension d h d_{h} to 128. The KV compres...

[domain:ev:bc7b716b6328] (paper, direct) chars:3000-6000+0
  TransMLA: Migrating GQA Models to MLA with Full DeepSeek Compatibility
  https://arxiv.org/html/2502.07864v4
  인용: ng KV cache compression Xiao et al. (2024) ; Liu et al. (2024b) ; Hooper et al. (2024) ; Zhang et al. (2023) r...


In [11]:
# 검색 로그: 어떤 질의가 어떤 출처를 데려왔고 무엇을 왜 걸렀는지 남는다.
log = out["search_log_by_perspective"]["domain"]
accepted = sum(len(entry["accepted_urls"]) for entry in log)
rejected = sum(len(entry["rejected"]) for entry in log)
print(f"질의 {len(log)}건 / 채택 {accepted} / 거부 {rejected}")

for entry in log[:3]:
    print(f"\nQ: {entry['query'][:70]}  (캐시={entry['from_cache']})")
    for url in entry["accepted_urls"][:2]:
        print(f"   채택 {url[:70]}")
    for item in entry["rejected"][:2]:
        print(f"   거부 {item['url'][:60]} <- {item['reason']}")

질의 12건 / 채택 70 / 거부 2

Q: How does DeepSeek-V2 MLA's compression of KV cache to low-dimensional   (캐시=False)
   채택 https://arxiv.org/pdf/2405.04434
   채택 https://arxiv.org/html/2502.07864v4

Q: What are the potential limitations or trade-offs of using DeepSeek-V2   (캐시=True)
   채택 https://openreview.net/pdf?id=JHvS2Q9RtW
   채택 https://openreview.net/pdf/72880f88aa934ab29407620344b11d7e3c598f3e.pd

Q: What is the reported throughput improvement when using ITME with CXL-H  (캐시=False)
   채택 https://dl.acm.org/doi/10.1145/3639564
   채택 https://dl.acm.org/doi/pdf/10.1145/3757347.3759131


In [12]:
# 관점 격리 검증: 미끼로 넣은 시장·이해관계자 문장이 결과에 새지 않았는지 확인한다.
blob = json.dumps(out, ensure_ascii=False)
leaks = [s for s in ("시장 규모가 급성장", "업계가 CXL을 반긴다") if s in blob]
print("관점 격리:", "누수 없음" if not leaks else f"누수 {leaks}")

# State 저장 경계: JSON 직렬화 가능해야 체크포인터에 들어간다.
print(f"직렬화 크기: {len(blob):,} bytes")

관점 격리: 누수 없음
직렬화 크기: 53,322 bytes


## 5. 재현성 기록

보고서에 인용한 수치를 나중에 되짚으려면 그 결과가 어떤 조건에서 나왔는지 남아야 합니다.
모델 버전이나 검색 캐시가 바뀌면 같은 질문에도 다른 답이 나오기 때문입니다.
산출물은 팀 규칙(ISSUE.md 5-1)에 따라 `outputs/domain/` 아래에 남깁니다.

In [13]:
out_dir = ROOT / "outputs/domain"
out_dir.mkdir(parents=True, exist_ok=True)
result_path = out_dir / "domain_findings_demo.json"
result_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")

manifest.record_artifact(file_checksum(result_path))
ablation_path = out_dir / "ablation.json"
if ablation_path.exists():
    manifest.record_artifact(file_checksum(ablation_path))
manifest_path = manifest.finish().save(out_dir / "run_manifest.json")

print("LLM      :", manifest.llm)
print("임베딩   :", manifest.embedding)
print("하드웨어 :", {k: manifest.hardware.get(k) for k in ("os", "machine", "ram_gb", "torch_backend")})
print("git      :", (manifest.git_commit or "")[:12])
print("\n산출물 체크섬")
for a in manifest.artifacts:
    print(f"  {a['name']:<28} {a['sha256'][:16]}  {a['bytes']:,}B")
print(f"\n저장: {manifest_path}")

LLM      : {'provider': 'openai', 'model': 'gpt-4o', 'temperature': 0.0, 'max_tokens': None, 'seed': None, 'prompt_version': 'domain/v3.2-single-prompt-quality'}
임베딩   : {'model_name': 'BAAI/bge-m3', 'device': 'mps:0', 'normalize_embeddings': True, 'batch_size': 16, 'dimension': 1024, 'torch_version': '2.14.0', 'platform': 'Darwin arm64'}
하드웨어 : {'os': 'Darwin 25.5.0', 'machine': 'arm64', 'ram_gb': 16.0, 'torch_backend': 'mps'}
git      : 99ac77366c13

산출물 체크섬
  domain_findings_demo.json    3e14ff4927109edf  65,506B

저장: /Users/sanlee/Desktop/SKALA/코딩파일/Ai-service/Capstone_Ai_RAG/outputs/domain/run_manifest.json


## 6. 검색 방식 비교 결과

`python -m agents.domain.tools.ablation --cache data/fetch_cache --embedding BAAI/bge-m3`
로 재현합니다(기본 산출물 경로는 `outputs/domain/`). Golden Set 은 정답을 청크 ID 가
아니라 "반드시 들어 있어야 할 표지 문자열"로 정의했습니다. 청킹 파라미터를 바꿔도
같은 기준으로 비교하기 위해서입니다.

In [14]:
ablation_path = ROOT / "outputs/domain/ablation.json"
if ablation_path.exists():
    report = json.loads(ablation_path.read_text(encoding="utf-8"))
    print(f"청크 {report['chunk_count']}개 / 질의 {report['golden_set_size']}개")
    header = f"{'method':<26}{'Hit@1':>7}{'Hit@3':>7}{'Hit@5':>7}{'MRR':>7}{'mean ms':>9}{'peak MB':>9}"
    print(header)
    print("-" * len(header))
    for row in report["results"]:
        print(f"{row['method']:<26}{row['hit_at_1']:>7}{row['hit_at_3']:>7}"
              f"{row['hit_at_5']:>7}{row['mrr']:>7}{row['mean_latency_ms']:>9}{row['peak_memory_mb']:>9}")
else:
    print("ablation.json 없음 — python -m agents.domain.tools.ablation 을 먼저 실행하세요")

ablation.json 없음 — python -m agents.domain.tools.ablation 을 먼저 실행하세요


## 정리와 남은 위험

- 검색 대상은 공신력 등급표에 있는 웹 출처뿐이며, 제공자 필터를 통과한 결과도 다시 거릅니다.
- 임베딩은 긴 문서 색인에만 쓰입니다. 짧은 문서는 색인 없이 그대로 근거가 됩니다.
- 근거가 부족하면 메우지 않고 `unknown` 과 `gaps` 로 남깁니다.
- 품질 검증은 이제 단일 프롬프트(self_check)가 맡습니다. 코드에 남은 것은 참조
  무결성뿐이라, 근거-대상 불일치 같은 의미적 오류는 프롬프트 지시에 의존합니다.

남은 위험:
- hybrid 융합 가중치를 조정하지 않았습니다. 측정에서는 dense 단독이 MRR 기준 앞섭니다.
- Golden Set 이 16문항이라 한 문항이 지표의 0.06을 좌우합니다.
- 200페이지 한도는 런타임 누적으로 막습니다. 어떤 페이지가 들어갔는지는 실행 후
  `search_log` 와 `evidence_store` 로만 확인됩니다.
- 단일 프롬프트 품질 검증은 결정적 코드보다 일관성이 낮을 수 있습니다. 실행마다
  `quality_by_perspective`의 `violations`/`warnings`를 확인해야 합니다.